In [1]:
from dotenv import load_dotenv
load_dotenv()

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model = 'gemini-1.5-flash-8b',
    temperature = 0,
    max_tokens = None,
    timeout = None,
    max_retries = 2,
)


I0000 00:00:1733351274.731446 10302201 check_gcp_environment_no_op.cc:29] ALTS: Platforms other than Linux and Windows are not supported


In [51]:
# checking what alternative would be if we just extracted text using a standard OCR
import fitz
from PIL import Image
import io
import base64

def extract_pdf_content(file_path: str) -> tuple[str, list[str]]:
    """
    Extract text and images from a PDF file and encode images in Base64.
    """
    pdf_content = ""
    images_base64 = []

    with fitz.open(file_path) as pdf:
        for page in pdf:
            pdf_content += page.get_text()

            for img in page.get_images(full=True):
                xref = img[0]
                base_image = pdf.extract_image(xref)
                image_bytes = base_image["image"]
                image = Image.open(io.BytesIO(image_bytes))
                buffered = io.BytesIO()
                image.save(buffered, format="PNG")
                images_base64.append(base64.b64encode(buffered.getvalue()).decode("utf-8"))

    print(f"Extracted PDF content: {pdf_content}")
    print(f"Extracted images: {len(images_base64)} images")

    return pdf_content, images_base64

pdf_content, images_base64 = extract_pdf_content('Notes/2024-08-27-ExSimplex.pdf')



Extracted PDF content:  
Simplex Method
v
p 20
mar
37
2
2
subject to
372
12
it
8
2X
10
111
270
Slack variable form
W
12
41
3
2
Wz
8
X
X2
W
10
2
1 42
X1
X2
X
Wi
Wz Wz
0
eg axtby
so
to
ax
by
0
axtby
so
2
0
eg axtby
w
who
to
ax
by
W
w
o
axt by
W
W 20
2
0
FO
it 3 2
12
WFO
2
5
2
10
Wz
O
2
0
1
2 8
W2 0
FO
1
3 2
12
w
O
E
w
o wz
o
wed wed
not feasible
x 0,4
0
2
5
2
10
Wz
O
B
W 0,43
0
A
MIX o
2
0
1
2 8
1
2
0 Wz 0
Wz 0
O
max J
3
2
2
subject to
W
12
41
3
2
Wz
8
X
X2
W
10
2x
X2
X1
X2
X
Wi
Wz Wz
0
X1
X
X
on the RHS
non basic variables
Wi
Wz Wz
on the LHS
basic variable
Idea ofSimplex Method
1
set all
nonbasic val.to
Zero
correspondto
a vertex
choose
a
new
set of
non basic
vars
to improve
II
2
is the
same
as moving from vertex to vertex
Initial step Guess
max J
3
1
2
2
subject to
W
12
71
3
2
Wz
8
X
X2
Wz
10
2
1
12
X1
X2
X
Wi
Wz
W
0
Set
0
10
3
0
Improve 0
71 0 12
0
max J
make
keep
max coeff rule
subject to
W
12
41
3
2
Wz
8
X
X2
Wz
10
2
1
12
X1
X2
X
Wi
Wz
W
0
W
X
can be
as large as possible
Ws
8
X
30


In [4]:
from langchain_core.messages import HumanMessage, AIMessage
import base64
import io
import os
from pdf2image import convert_from_path

def encode_image(image):
    """Convert image to base64 string"""
    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    image_bytes = buffer.getvalue()
    return base64.b64encode(image_bytes).decode('utf-8')

def process_slides(num_docs=None, num_slides=None, output_dir='./output_MA351', overwrite=False):
    """Process slides sequentially with context from previous generations"""
    
    os.makedirs(output_dir, exist_ok=True)
    notes_dir = '/Users/ashoksaravanan/Coding/ScribeLec/Server/summary/Notes_MA351/'
    
    # Filter for PDF files and validate them
    pdf_files = [f for f in os.listdir(notes_dir) if f.lower().endswith('.pdf')]
    if num_docs:
        pdf_files = pdf_files[:num_docs]
    
    responses = []
    
    for pdf_file in pdf_files:
        try:
            pdf_path = os.path.join(notes_dir, pdf_file)
            
            # Validate PDF file
            if not os.path.isfile(pdf_path):
                print(f"Skipping {pdf_file} - not a valid file")
                continue
                
            # Check file size
            if os.path.getsize(pdf_path) == 0:
                print(f"Skipping {pdf_file} - empty file")
                continue
            
            pdf_output_dir = os.path.join(output_dir, pdf_file.replace('.pdf', ''))
            
            try:
                images = convert_from_path(pdf_path)
                if num_slides is not None and len(images) > num_slides:
                    images = images[:num_slides]
                
                # Check if all slides already exist
                all_slides_exist = True
                for page_number in range(1, len(images) + 1):
                    slide_file = os.path.join(pdf_output_dir, f"{page_number}.txt")
                    if not os.path.exists(slide_file) or overwrite:
                        all_slides_exist = False
                        break
                        
                if all_slides_exist and not overwrite:
                    print(f"Skipping {pdf_file} - all slides already processed")
                    continue
                    
                # Create subdirectory for PDF file
                os.makedirs(pdf_output_dir, exist_ok=True)
                
                # Base prompt
                base_prompt = f"Extract exactly what is written on the lecture notes, in the context of Linear Programming. Output the content in LaTeX format, preserving the formatting of the slide. For any figures you see, try to re-create them in LaTeX. Take note of direction of arrows, placement of labels, and other notations. Below the generation, provide a description of what you see: include specific details that would not be known unless you were given the context of the slide."
                
                conversation_history = []
                pdf_responses = []
                
                for image_file, page_number in zip(images, range(len(images))):
                    page_number += 1
                    # Check if individual slide file exists
                    slide_file = os.path.join(pdf_output_dir, f"{page_number}.txt")
                    if os.path.exists(slide_file) and not overwrite:
                        print(f"Skipping slide {page_number} - output already exists")
                        # Read existing response to maintain context
                        with open(slide_file, "r") as f:
                            current_response = f.read()
                        pdf_responses.append(current_response)
                        conversation_history.extend([
                            HumanMessage(content=[{"type": "text", "text": "Previous slide content"}]),
                            AIMessage(content=current_response)
                        ])
                        continue
                        
                    additional_prompt = f"This is slide {page_number} of {len(images)} slides. Use the previous slide's generation to help you understand the context of the current slide. Output the slide number at the top of your response for clarity."
                    
                    # Encode current image
                    image_base64 = encode_image(image_file)
                    
                    # Create message with context from previous responses
                    message_content = [
                        {"type": "text", "text": base_prompt + "\n\n" + additional_prompt}
                    ]
                    
                    # Add previous responses as context
                    if conversation_history:
                        context = "\n\nPrevious slides contained:\n" + "\n".join(
                            [f"Slide {i+1}: {resp}" for i, resp in enumerate(pdf_responses)]
                        )
                        message_content[0]["text"] += context
                        
                    # Add current image
                    message_content.append({
                        "type": "image_url",
                        "image_url": f"data:image/png;base64,{image_base64}"
                    })
                    
                    # Create message and get response
                    message = HumanMessage(content=message_content)
                    conversation_history.append(message)
                    
                    response = llm.generate([conversation_history])
                    current_response = response.generations[0][0].text
                    pdf_responses.append(current_response)
                    
                    # Add AI response to conversation history
                    conversation_history.append(AIMessage(content=current_response))
                    
                    print(f"\nProcessing Slide {page_number}:")
                    print(response)
                    # saving individual response in file
                    with open(slide_file, "w") as f:
                        f.write(current_response)
                        
                print(f"Processing {pdf_file} complete.")
                # saving concatenated response in file
                output_file = os.path.join(output_dir, f"{pdf_file.replace('.pdf', '')}.txt")
                with open(output_file, "w") as f:
                    f.write("\n\n".join(pdf_responses))
                
                responses.extend(pdf_responses)
                
            except Exception as e:
                print(f"Error processing PDF {pdf_file}: {str(e)}")
                continue
                
        except Exception as e:
            print(f"Error reading {pdf_file}: {str(e)}")
            continue

    return responses

# Process all slides
responses = process_slides()
print(responses)

Skipping 2024-10-10-BasisDim.pdf - all slides already processed
Skipping 2024-09-05-ColNullSpaces.pdf - all slides already processed
Skipping 2024-11-26-DiagAppls.pdf - all slides already processed
Skipping 2024-08-22-SolvingMNLinearSystems.pdf - all slides already processed
Skipping 2024-10-15-ColNullRow.pdf - all slides already processed
Skipping 2024-09-10-VectorSpace.pdf - all slides already processed
Skipping 2024-08-20-ThreeInterpretationsLinSys.pdf - all slides already processed
Skipping 2024-10-22-MatrixMult.pdf - all slides already processed
Skipping 2024-08-22-GaussianElimination.pdf - all slides already processed
Skipping 2024-09-19-LinDepLinInd.pdf - all slides already processed
[]


In [24]:
for response in responses:
    print(response)


Slide 1

\textbf{Network Flow (LV) Chapter 14)}

$N = \text{set of nodes } \{i, j\}$

$\mathcal{A} = \text{set of (directed) arcs}$

$\subseteq \{c_{ij}\} : i, j \in N$

\begin{tikzpicture}[scale=0.5]
\filldraw[black] (0,0) circle (2pt) node[below] {};
\filldraw[black] (2,2) circle (2pt) node[below] {$i$};
\filldraw[black] (4,0) circle (2pt) node[below] {};
\filldraw[black] (6,2) circle (2pt) node[below] {$j$};
\filldraw[black] (8,0) circle (2pt) node[below] {};

\draw[->] (2,2) -- (4,0);
\draw[->] (2,2) -- (6,2);
\draw[->] (6,2) -- (8,0);
\draw[->] (4,0) -- (6,2);
\end{tikzpicture}

Network = (N, $\mathcal{A}$)
Graph (or digraph)


\textbf{Description:}

This slide introduces the concept of Network Flow, specifically in the context of Linear Programming (as indicated by the "LV" and chapter number).  It defines the key components:

* **N:** The set of nodes (represented by dots in the diagram).  The handwritten notes use $i$ and $j$ as examples of nodes.
* **$\mathcal{A}$:** The set o

In [19]:
from PIL import Image
from langchain_core.messages import HumanMessage, AIMessage
import base64
import io

original_image = Image.open('../network-flow/page_3.png')
buffer = io.BytesIO()
original_image.save(buffer, format="PNG")
image_bytes = buffer.getvalue()
image_base64 = base64.b64encode(image_bytes).decode('utf-8')

output_image = Image.open('../network-flow/page_3_out.png')
buffer = io.BytesIO()
output_image.save(buffer, format="PNG")
image_bytes = buffer.getvalue()
output_image_base64 = base64.b64encode(image_bytes).decode('utf-8')

prompt = "Extract exactly what is written on the slide. Output the content in LaTeX format, preserving the formatting of the slide. For any figures you see, try to re-create them in LaTeX."

initial_message = HumanMessage(
    content=[
        {
            "type": "text", 
            "text": prompt
        },
        {
            "type": "image_url",
            "image_url": f"data:image/png;base64,{image_base64}"
        }
    ]
)

ai_message = AIMessage(
    content=[
        {
            "type": "text",
            "text": response.generations[0][0].text
        },
        {
            "type": "image_url",
            "image_url": f"data:image/png;base64,{output_image_base64}"
        },
    ]
)

final_message = HumanMessage(
    content=[
        {
            "type": "text",
            "text": "This is your text response and the corresponding LaTeX output. Make any corrections to your previous response to ensure that the output is correct."
        }
    ]
)

response = llm.generate([[initial_message, ai_message, final_message]])
print(response)

generations=[[ChatGeneration(text='```latex\n\\documentclass{article}\n\\usepackage{amsmath}\n\\usepackage{tikz}\n\n\\begin{document}\n\n\\section*{Network Flow (IV) Chapter 14}\n\n\\begin{tikzpicture}[scale=0.8]\n\\tikzstyle{every node}=[circle, draw, fill=black!25, minimum size=10pt, inner sep=0pt]\n\\node (i) at (0,1) {i};\n\\node (k) at (2,0.5) {k};\n\\node (j) at (4,1) {j};\n\\draw (i) -- (k) node[midway, above] {$b_k$};\n\\draw (k) -- (j);\n\\end{tikzpicture}\n\n\\textbf{Balanced eqn at node k:}\n\\begin{equation*}\n\\sum_{i} x_{ik} + b_k = \\sum_{j} x_{kj}\n\\end{equation*}\n\n\\begin{equation*}\n\\sum_{i} x_{ik} - \\sum_{j} x_{kj} = -b_k\n\\end{equation*}\n\n\\begin{equation*}\n\\min \\ c^T x \\\\\nAx = -b, \\quad x \\ge 0\n\\end{equation*}\n\n\\end{document}\n```\n\n**Explanation of Improvements:**\n\n1. **TikZ for Diagram:** The previous response lacked the crucial TikZ code to create the graph. This improved version uses TikZ to draw the nodes (circles) and the edges (lines)

In [20]:
print(response.generations[0][0].text)

```latex
\documentclass{article}
\usepackage{amsmath}
\usepackage{tikz}

\begin{document}

\section*{Network Flow (IV) Chapter 14}

\begin{tikzpicture}[scale=0.8]
\tikzstyle{every node}=[circle, draw, fill=black!25, minimum size=10pt, inner sep=0pt]
\node (i) at (0,1) {i};
\node (k) at (2,0.5) {k};
\node (j) at (4,1) {j};
\draw (i) -- (k) node[midway, above] {$b_k$};
\draw (k) -- (j);
\end{tikzpicture}

\textbf{Balanced eqn at node k:}
\begin{equation*}
\sum_{i} x_{ik} + b_k = \sum_{j} x_{kj}
\end{equation*}

\begin{equation*}
\sum_{i} x_{ik} - \sum_{j} x_{kj} = -b_k
\end{equation*}

\begin{equation*}
\min \ c^T x \\
Ax = -b, \quad x \ge 0
\end{equation*}

\end{document}
```

**Explanation of Improvements:**

1. **TikZ for Diagram:** The previous response lacked the crucial TikZ code to create the graph. This improved version uses TikZ to draw the nodes (circles) and the edges (lines) representing the network flow.  The `\tikzstyle{every node}=[...]` part styles the nodes to be circles